In [2]:
import pandas as pd
import numpy as np
import os
import sys
import ast
import re
import json

## Sample XX Replicates per Polymer Distribution

In [3]:
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..'))
sys.path.append(project_root)

from utils.generate_seq import Sample

In [4]:
num_replicates = 100 # How many replicates per polymer distribution

df = pd.read_csv('polymer_db.csv')
df[['monomers', 'mon_mol_distribution']] = df[['monomers', 'mon_mol_distribution']].map(lambda x: ast.literal_eval(str(x)))

# for col in ['monomers', 'mon_SMILES', 'mon_class_wt_%', 'mon_mol_distribution']:
#     print(f"Column: {col}")
#     print(df[col].apply(type).value_counts())

In [5]:
def generate_samples_for_row(row):
    monomers = row['monomers']
    mol_dist = row['mon_mol_distribution']
    DP = row['DP']
    n= num_replicates
    
    return Sample(monomers, DP, mol_dist, 'wo_replacement', n, batch=True).samples

# Generate samples and replicate the data for each row in a more memory-efficient manner
samples_list = df.apply(generate_samples_for_row, axis=1)

new_df = df.loc[df.index.repeat(num_replicates), ['poly_ID', 'monomers', 'MIC_ecoli']].reset_index(drop=True)
new_df['sequence'] = [sample for samples in samples_list for sample in samples]

new_df

,poly_ID,monomers,MIC_ecoli,sequence
0,1,"[Tma, Ni, Mo]",900.810811,TmaTmaTmaNiTmaTmaTmaTmaTmaNiTmaTmaTmaTmaTmaTma...
1,1,"[Tma, Ni, Mo]",900.810811,NiNiNiTmaNiTmaTmaMoNiTmaTmaNiMoTmaTmaMoMoTmaTm...
2,1,"[Tma, Ni, Mo]",900.810811,NiNiTmaTmaMoTmaNiNiTmaMoNiTmaTmaNiTmaNiNiNiNiT...
3,1,"[Tma, Ni, Mo]",900.810811,NiTmaNiNiNiTmaTmaTmaTmaTmaTmaNiTmaNiNiNiNiNiNi...
4,1,"[Tma, Ni, Mo]",900.810811,TmaNiTmaNiTmaNiTmaNiNiNiTmaMoTmaTmaMoNiNiNiTma...
...,...,...,...,...
5195,52,"[Tma, Mep, Phe, Olam, Bmam]",2.471042,BmamTmaTmaBmamTmaTmaTmaBmamOlamTmaTmaOlamOlamT...
5196,52,"[Tma, Mep, Phe, Olam, Bmam]",2.471042,TmaTmaTmaTmaTmaTmaTmaTmaTmaBmamTmaTmaBmamPheTm...
5197,52,"[Tma, Mep, Phe, Olam, Bmam]",2.471042,TmaTmaTmaTmaTmaPheTmaTmaBmamMepBmamTmaOlamTmaT...
5198,52,"[Tma, Mep, Phe, Olam, Bmam]",2.471042,BmamMepPheTmaTmaTmaTmaTmaTmaTmaTmaTmaTmaOlamTm...


## Calculate Molar Ratio of Monomers in Sampled Sequence

In [6]:
with open('monomer_data/monomer_properties.json', 'r') as file:
    monomer_properties = json.load(file)

def get_polymer_weights(seq):

    seq = re.findall('[A-Z][^A-Z]*', seq)
    freq = {}
    
    for mon in seq:
        if (mon in freq):
            freq[mon] += monomer_properties[mon]['molar_mass']
        else:
            freq[mon] = monomer_properties[mon]['molar_mass']

    return freq

def get_seq_dist(seq):
    wts = get_polymer_weights(seq)
    total_wts = sum(list(wts.values()))
    wts = {key: val / total_wts for key, val in wts.items()}

    return wts

In [7]:
new_df['seq_mon_mol_dist'] = new_df['sequence'].map(lambda x: get_seq_dist(x))
new_df['seq_mon_mol_dist'] = new_df.apply(lambda row:
                                               np.array([row['seq_mon_mol_dist'].get(mon, 0) for mon in row['monomers'] if mon in row['seq_mon_mol_dist']]), axis=1)

In [8]:
new_df = new_df.drop('monomers', axis=1)
# new_df.insert(0, 'sample_ID', range(1, len(new_df) + 1))
new_df.insert(0, 'sample_ID', (np.array([i for i in range(0, len(new_df))]) % (num_replicates)) + 1)
new_df

,sample_ID,poly_ID,MIC_ecoli,sequence,seq_mon_mol_dist
0,1,1,900.810811,TmaTmaTmaNiTmaTmaTmaTmaTmaNiTmaTmaTmaTmaTmaTma...,"[0.6479333604409899, 0.2660253703286092, 0.086..."
1,2,1,900.810811,NiNiNiTmaNiTmaTmaMoNiTmaTmaNiMoTmaTmaMoMoTmaTm...,"[0.6320004123156591, 0.25701242362988025, 0.11..."
2,3,1,900.810811,NiNiTmaTmaMoTmaNiNiTmaMoNiTmaTmaNiTmaNiNiNiNiT...,"[0.5492443365325148, 0.3214109993815424, 0.129..."
3,4,1,900.810811,NiTmaNiNiNiTmaTmaTmaTmaTmaTmaNiTmaNiNiNiNiNiNi...,"[0.5998744880672977, 0.3489159178876268, 0.051..."
4,5,1,900.810811,TmaNiTmaNiTmaNiTmaNiNiNiTmaMoTmaTmaMoNiNiNiTma...,"[0.6527088756815168, 0.2977623094033526, 0.049..."
...,...,...,...,...,...
5195,96,52,2.471042,BmamTmaTmaBmamTmaTmaTmaBmamOlamTmaTmaOlamOlamT...,"[0.5587555546703334, 0.05092477658612647, 0.07..."
5196,97,52,2.471042,TmaTmaTmaTmaTmaTmaTmaTmaTmaBmamTmaTmaBmamPheTm...,"[0.7532262907349266, 0.04173845108893692, 0.03..."
5197,98,52,2.471042,TmaTmaTmaTmaTmaPheTmaTmaBmamMepBmamTmaOlamTmaT...,"[0.8068519713776292, 0.03991964769893263, 0.03..."
5198,99,52,2.471042,BmamMepPheTmaTmaTmaTmaTmaTmaTmaTmaTmaTmaOlamTm...,"[0.7009816961557329, 0.0431593630391812, 0.077..."


In [9]:
new_df.to_csv('polymer_samples_db.csv', index=False)